# Cylinder Stroke Time Series — Gaussian Process Analysis

## 1 — Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import GPy
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams.update({
    "figure.facecolor":  "white",
    "axes.facecolor":    "#F8F8F6",
    "axes.grid":         True,
    "grid.color":        "#E0DED6",
    "grid.linewidth":    0.6,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.family":       "sans-serif",
    "axes.titlesize":    12,
    "axes.labelsize":    11,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
})

print(f"GPy {GPy.__version__}  ready")

## 2 — Dataset prep

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────
CSV_PATH          = "./datasets/cylinder_strokes.csv"
SLOW_THRESHOLD_MS = 500
# ───────────────────────────────────────────────────────────────────

df = pd.read_csv(CSV_PATH)
df["date"]     = pd.to_datetime(df["date"])
df["datetime"] = df["date"] + pd.to_timedelta(df["t_start_ms"], unit="ms")
df = df.sort_values("datetime").reset_index(drop=True)

start = '2026-03-05 16:00'
# end   = '2026-03-05 18:00'
# df = df[(df['datetime'] >= start) & (df['datetime'] < end)]


CYLINDERS  = sorted(df["cylinder"].unique().tolist())
DIRECTIONS = sorted(df["direction"].unique().tolist())

print(f"Rows      : {len(df):,}")
print(f"Date range: {df['datetime'].min()}  to  {df['datetime'].max()}")
print(f"Cylinders : {CYLINDERS}")
print(f"Directions: {DIRECTIONS}")
print()

summary = (
    df.groupby(["cylinder", "direction"])["duration_ms"]
    .agg(
        n         = "count",
        mean      = "mean",
        std       = "std",
        p50       = lambda x: x.quantile(0.50),
        p95       = lambda x: x.quantile(0.95),
        slow_pct  = lambda x: (x > SLOW_THRESHOLD_MS).mean() * 100,
    )
    .round(1)
)
display(summary)

## 3 — EDA

Three views, all working directly on raw events:
- **Duration scatter** — every stroke plotted against time, per cylinder/direction
- **Actuation rate** — strokes per hour over time (are cylinders working harder or less?)
- **Duration distributions** — shape of the duration spread per cylinder/direction

In [ ]:
# ── EDA 1: Duration scatter ─────────────────────────────────────────

combos = [
    (cyl, dirn)
    for cyl in CYLINDERS for dirn in DIRECTIONS
    if len(df[(df["cylinder"] == cyl) & (df["direction"] == dirn)]) > 0
]

ncols = 2
nrows = int(np.ceil(len(combos) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.ravel()

for i, (cyl, dirn) in enumerate(combos):
    sub = df[(df["cylinder"] == cyl) & (df["direction"] == dirn)]
    axes[i].scatter(
        sub["datetime"], sub["duration_ms"],
        s=1.5, alpha=0.2, color="#378ADD", rasterized=True,
    )
    axes[i].axhline(
        SLOW_THRESHOLD_MS, color="#D85A30", lw=0.8,
        linestyle="--", alpha=0.7, label=f"slow > {SLOW_THRESHOLD_MS}ms",
    )
    axes[i].set_title(f"{cyl} / {dirn}", fontsize=11)
    axes[i].set_ylabel("duration (ms)")
    axes[i].legend(fontsize=7, loc="upper right")
    loc = mdates.AutoDateLocator()
    axes[i].xaxis.set_major_locator(loc)
    axes[i].xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Raw stroke durations over time", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── EDA 2: Actuation rate — strokes per hour per cylinder ───────────

fig, axes = plt.subplots(len(CYLINDERS), 1, figsize=(15, 3.5 * len(CYLINDERS)))
if len(CYLINDERS) == 1:
    axes = [axes]

dir_colors = dict(zip(DIRECTIONS, ["#378ADD", "#D85A30", "#1D9E75", "#9B59B6"]))

for ax, cyl in zip(axes, CYLINDERS):
    for dirn in DIRECTIONS:
        sub = df[(df["cylinder"] == cyl) & (df["direction"] == dirn)].copy()
        if len(sub) == 0:
            continue
        rate = sub.groupby(sub["datetime"].dt.floor("h")).size()
        ax.plot(rate.index, rate.values,
                color=dir_colors[dirn], lw=1.0, alpha=0.85, label=dirn)
    ax.set_title(f"{cyl} — strokes per hour", fontsize=11)
    ax.set_ylabel("count")
    ax.legend(fontsize=8, loc="upper right")
    loc = mdates.AutoDateLocator()
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))

axes[-1].set_xlabel("datetime")
fig.suptitle("Actuation rate over time", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── EDA 3: Duration distributions ───────────────────────────────────

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.ravel()

for i, (cyl, dirn) in enumerate(combos):
    sub = df[(df["cylinder"] == cyl) & (df["direction"] == dirn)]
    axes[i].hist(
        sub["duration_ms"], bins=60,
        color="#378ADD", edgecolor="white", linewidth=0.3, alpha=0.85,
    )
    axes[i].axvline(
        SLOW_THRESHOLD_MS, color="#D85A30",
        lw=1.2, linestyle="--", label="slow threshold",
    )
    axes[i].axvline(
        sub["duration_ms"].median(), color="#1D9E75",
        lw=1.2, linestyle=":",
        label=f"median = {sub['duration_ms'].median():.0f}ms",
    )
    slow_pct = (sub["duration_ms"] > SLOW_THRESHOLD_MS).mean() * 100
    axes[i].set_title(f"{cyl} / {dirn}  —  slow = {slow_pct:.1f}%", fontsize=11)
    axes[i].set_xlabel("duration (ms)")
    axes[i].set_ylabel("count")
    axes[i].legend(fontsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Duration distributions per cylinder / direction", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4 — GP model

`run_gp()` fits a **Sparse Gaussian Process** directly on raw stroke events — no binning,
no gap-filling. Each event is an observation `(timestamp, duration_ms)`.

`show_results()` produces a 3-panel diagnostic figure and a plain-English summary.

### Kernel options

| String | Behaviour | When to use |
|---|---|---|
| `"rbf"` | Very smooth | Clean, slowly-drifting signal |
| `"matern52"` | Moderately rough | **Default — good for noisy industrial data** |
| `"matern32"` | Rough | High-frequency noise |
| `"periodic+rbf"` | Cycle + trend | Suspected shift or daily rhythm |

In [ ]:
def run_gp(
    cylinder:    str,
    direction:   str,
    train_start: str,
    train_end:   str,
    test_end:    str,
    kernel:      str  = "matern52",
    n_inducing:  int  = None,
    n_restarts:  int  = 3,
    verbose:     bool = True,
) -> dict:
    """
    Fit a Sparse GP on raw cylinder stroke events.

    Parameters
    ----------
    cylinder, direction
        Which cylinder and stroke direction to model.

    train_start, train_end, test_end : str (datetime-parseable)
        train window : [train_start, train_end)
        test window  : [train_end,   test_end)

    kernel : str
        "rbf" | "matern52" | "matern32" | "periodic+rbf"

    n_inducing : int or None
        Inducing points for the sparse approximation.
        None = auto: clip(n_train // 10, 50, 500).
        More inducing points = more accurate, slower to fit.

    n_restarts : int
        Optimiser restarts. More = less likely to land in a local optimum.

    Returns
    -------
    dict:
        cylinder, direction, kernel_name,
        events_train, events_test   — raw event DataFrames
        t_train, y_train            — numeric time (seconds from t0) + durations
        t_test,  y_test
        t0                          — datetime anchor (t = 0)
        model                       — fitted GPy SparseGPRegression
        forecast_mean, forecast_var — GP predictions at test event times
        conf_int                    — (n_test, 2) lower / upper 95% CI
        mae, rmse, mape
        n_inducing
    """
    # ── Filter & sort ────────────────────────────────────────────────
    sub = df[(df["cylinder"] == cylinder) & (df["direction"] == direction)].copy()
    if len(sub) == 0:
        raise ValueError(f"No data for cylinder='{cylinder}', direction='{direction}'")
    sub = sub.sort_values("datetime").reset_index(drop=True)

    # ── Train / test split ───────────────────────────────────────────
    events_train = sub[(sub["datetime"] >= train_start) & (sub["datetime"] < train_end)].copy()
    events_test  = sub[(sub["datetime"] >= train_end)   & (sub["datetime"] < test_end)].copy()

    if len(events_train) == 0:
        raise ValueError("Training set is empty — check your time window.")
    if len(events_test) == 0:
        raise ValueError("Test set is empty — check your time window.")

    # ── Timestamps → numeric seconds ────────────────────────────────
    # The GP needs a numeric X axis. We anchor at the first training
    # event so values stay in the hundreds/thousands range, which
    # is more numerically stable than raw Unix epoch timestamps.
    t0 = events_train["datetime"].iloc[0]
    def to_sec(dt_col):
        return (dt_col - t0).dt.total_seconds().values.reshape(-1, 1)

    # t_train = to_sec(events_train["datetime"])
    t_scale = (events_train["datetime"].iloc[-1] - t0).total_seconds()
    t_train = to_norm(events_train["datetime"])  # values 0.0–1.0
    y_train = events_train["duration_ms"].values.reshape(-1, 1)
    t_test  = to_sec(events_test["datetime"])
    y_test  = events_test["duration_ms"].values

    # ── Inducing points ──────────────────────────────────────────────
    # Standard GP scales as O(n^3) in the number of observations —
    # unusable at thousands of events. Sparse GP summarises the data
    # through m << n inducing points Z, reducing cost to O(n * m^2).
    # We space Z evenly across the training time window.
    n_train = len(t_train)
    if n_inducing is None:
        n_inducing = int(np.clip(n_train // 10, 50, 500))
    n_inducing = min(n_inducing, n_train)
    Z = np.linspace(t_train.min(), t_train.max(), n_inducing).reshape(-1, 1)

    # ── Kernel ───────────────────────────────────────────────────────
    kernel_map = {
        "rbf":          GPy.kern.RBF(1),
        "matern52":     GPy.kern.Matern52(1),
        "matern32":     GPy.kern.Matern32(1),
        "periodic+rbf": GPy.kern.StdPeriodic(1) + GPy.kern.RBF(1),
    }
    if kernel not in kernel_map:
        raise ValueError(f"kernel must be one of {list(kernel_map)}")

    # ── Fit ──────────────────────────────────────────────────────────
    model = GPy.models.SparseGPRegression(t_train, y_train,
                                          kernel=kernel_map[kernel], Z=Z)
    # Initialise noise variance to ~10% of signal variance
    model.likelihood.variance = float(np.var(y_train)) * 0.1

    if verbose:
        print(f"\n{'='*60}")
        print(f"  {cylinder} / {direction}  [{kernel}]")
        print(f"  Train : {events_train['datetime'].iloc[0]}")
        print(f"        → {events_train['datetime'].iloc[-1]}  ({n_train:,} events)")
        print(f"  Test  : {events_test['datetime'].iloc[0]}")
        print(f"        → {events_test['datetime'].iloc[-1]}  ({len(events_test):,} events)")
        print(f"  Inducing pts : {n_inducing}")
        print(f"{'='*60}")
        print("Optimising...")

    model.optimize_restarts(
        num_restarts=n_restarts,
        verbose=False, messages=False, robust=True,
    )

    # ── Predict at test event times ──────────────────────────────────
    mean_pred, var_pred = model.predict(t_test)
    mean_pred = mean_pred.ravel()
    std_pred  = np.sqrt(np.clip(var_pred.ravel(), 0, None))
    conf_int  = np.column_stack([
        mean_pred - 1.96 * std_pred,
        mean_pred + 1.96 * std_pred,
    ])

    # ── Metrics ──────────────────────────────────────────────────────
    mae  = mean_absolute_error(y_test, mean_pred)
    rmse = np.sqrt(mean_squared_error(y_test, mean_pred))
    mape = np.mean(np.abs((y_test - mean_pred) / (y_test + 1e-9))) * 100

    if verbose:
        print(f"  MAE  = {mae:.2f} ms")
        print(f"  RMSE = {rmse:.2f} ms")
        print(f"  MAPE = {mape:.2f}%")

    return dict(
        cylinder=cylinder, direction=direction, kernel_name=kernel,
        events_train=events_train, events_test=events_test,
        t_train=t_train, y_train=y_train.ravel(),
        t_test=t_test,   y_test=y_test,
        t0=t0, model=model,
        forecast_mean=mean_pred,
        forecast_var=var_pred.ravel(),
        conf_int=conf_int,
        mae=mae, rmse=rmse, mape=mape,
        n_inducing=n_inducing,
    )


# ── Helpers used by show_results ────────────────────────────────────

def _to_dt(t0, t_sec):
    """Convert numeric seconds back to datetimes for plot axes."""
    return [t0 + pd.Timedelta(seconds=float(s)) for s in t_sec]


def _fmt_xaxis(ax):
    """Apply consistent concise date formatting to an axis."""
    loc = mdates.AutoDateLocator()
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))


def _print_summary(r: dict) -> None:
    """Print a plain-English interpretation of a run_gp() result."""
    model  = r["model"]
    kernel = r["kernel_name"]
    mae    = r["mae"]
    mape   = r["mape"]

    # Lengthscale — how far apart two events must be before the GP
    # treats them as uncorrelated
    try:
        ls = float(model.kern.lengthscale)
        ls_str = (
            f"{ls:.0f}s"        if ls < 120  else
            f"{ls/60:.1f}min"   if ls < 7200 else
            f"{ls/3600:.1f}h"
        )
        ls_msg = (
            f"Lengthscale = {ls_str}. "
            f"Strokes within ~{ls_str} of each other are treated as correlated."
        )
    except Exception:
        ls_msg = "(Compound kernel — inspect model.kern for lengthscale details.)"

    # Noise
    noise_std   = float(np.sqrt(model.likelihood.variance))
    noise_ratio = noise_std / (float(np.std(r["y_train"])) + 1e-9)
    noise_msg = (
        f"LOW ({noise_std:.1f}ms) — fitting the signal tightly."         if noise_ratio < 0.2 else
        f"MODERATE ({noise_std:.1f}ms) — typical for stroke duration data." if noise_ratio < 0.6 else
        f"HIGH ({noise_std:.1f}ms, {noise_ratio*100:.0f}% of signal std). "
        f"Very noisy signal or kernel mismatch."
    )

    # Accuracy
    accuracy_msg = (
        f"Excellent — MAPE {mape:.1f}%"                                          if mape < 3  else
        f"Good — MAPE {mape:.1f}%, MAE = {mae:.1f}ms"                            if mape < 8  else
        f"Moderate — MAPE {mape:.1f}%. Trend captured; individual events vary."  if mape < 20 else
        f"Weak — MAPE {mape:.1f}%. Focus on CI shape and trend, not point values."
    )

    print("\n" + "━" * 55)
    print(f"  {r['cylinder']} / {r['direction']}  [{kernel}]")
    print("━" * 55)
    print(f"  Kernel   : {kernel}")
    print(f"  {ls_msg}")
    print(f"  Noise    : {noise_msg}")
    print(f"  Accuracy : {accuracy_msg}")
    print(f"  MAE = {mae:.2f}ms   RMSE = {r['rmse']:.2f}ms   MAPE = {mape:.2f}%")
    print()
    print("  CI note  : CI width reflects data density — sparse periods")
    print("             get wider bands automatically.")
    if mape > 15:
        print()
        print("  ⚠  High MAPE — try a different kernel, or check whether")
        print("     the window spans multiple operating regimes.")
    if noise_ratio > 0.6:
        print()
        print("  ⚠  High noise — consider a longer training window.")
    print("━" * 55)


def show_results(r: dict) -> None:
    """
    3-panel diagnostic figure + plain-English summary for a run_gp() result.

    Panel A (top)          — full view: train events, GP mean, 95% CI, test events
    Panel B (bottom-left)  — test zoom: actual vs predicted with CI
    Panel C (bottom-right) — residual distribution
    """
    t0        = r["t0"]
    t_train   = r["t_train"].ravel()
    y_train   = r["y_train"]
    t_test    = r["t_test"].ravel()
    y_test    = r["y_test"]
    mean_pred = r["forecast_mean"]
    conf_int  = r["conf_int"]
    model     = r["model"]

    dt_train = _to_dt(t0, t_train)
    dt_test  = _to_dt(t0, t_test)

    # Dense grid for smooth GP mean / CI line across both windows
    t_all    = np.concatenate([t_train, t_test])
    t_grid   = np.linspace(t_all.min(), t_all.max(), 500).reshape(-1, 1)
    mu_grid, var_grid = model.predict(t_grid)
    mu_grid  = mu_grid.ravel()
    std_grid = np.sqrt(np.clip(var_grid.ravel(), 0, None))
    dt_grid  = _to_dt(t0, t_grid.ravel())

    residuals = y_test - mean_pred

    fig = plt.figure(figsize=(15, 10))
    gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.28)
    ax_main = fig.add_subplot(gs[0, :])
    ax_test = fig.add_subplot(gs[1, 0])
    ax_res  = fig.add_subplot(gs[1, 1])

    # Panel A
    ax_main.scatter(dt_train, y_train,
                    s=2, alpha=0.2, color="#378ADD",
                    label="Train events", rasterized=True)
    ax_main.scatter(dt_test, y_test,
                    s=5, alpha=0.45, color="#1D9E75",
                    label="Test events", zorder=3)
    ax_main.plot(dt_grid, mu_grid,
                 color="#D85A30", lw=1.8, label="GP mean", zorder=4)
    ax_main.fill_between(
        dt_grid,
        mu_grid - 1.96 * std_grid,
        mu_grid + 1.96 * std_grid,
        color="#D85A30", alpha=0.12, label="95% CI",
    )
    ax_main.axvline(_to_dt(t0, [t_test.min()])[0],
                    color="#888", lw=0.9, linestyle="--", alpha=0.6)
    ax_main.set_title(
        f"{r['cylinder']} / {r['direction']}  [kernel: {r['kernel_name']}]",
        fontsize=12)
    ax_main.set_ylabel("duration (ms)")
    ax_main.legend(fontsize=8, loc="upper left")
    _fmt_xaxis(ax_main)

    # Panel B
    ax_test.scatter(dt_test, y_test,
                    s=8, alpha=0.5, color="#1D9E75",
                    label="Actual", zorder=3)
    ax_test.plot(dt_test, mean_pred,
                 color="#D85A30", lw=1.5, label="GP mean", zorder=4)
    ax_test.fill_between(dt_test, conf_int[:, 0], conf_int[:, 1],
                         color="#D85A30", alpha=0.15, label="95% CI")
    ax_test.set_title("Test window — actual vs predicted", fontsize=11)
    ax_test.set_ylabel("duration (ms)")
    ax_test.legend(fontsize=8)
    _fmt_xaxis(ax_test)

    # Panel C
    ax_res.hist(residuals, bins=30,
                color="#378ADD", edgecolor="white", linewidth=0.4, alpha=0.85)
    ax_res.axvline(0,                color="#D85A30", lw=1.2, linestyle="--")
    ax_res.axvline(residuals.mean(), color="#888",   lw=1.0, linestyle=":")
    ax_res.set_title("Residuals  (actual − predicted)", fontsize=11)
    ax_res.set_xlabel("residual (ms)")
    ax_res.set_ylabel("count")
    ax_res.text(0.97, 0.95,
                f"bias = {residuals.mean():+.1f}\nstd = {residuals.std():.1f}",
                transform=ax_res.transAxes, ha="right", va="top",
                fontsize=8, color="#444")

    fig.suptitle(
        f"MAE = {r['mae']:.1f}ms   RMSE = {r['rmse']:.1f}ms   MAPE = {r['mape']:.1f}%",
        fontsize=11, y=1.01,
    )
    plt.tight_layout()
    plt.show()

    _print_summary(r)


print("✓  run_gp() and show_results() defined")

## 5a — GP Forecasting

Fit on a training window, evaluate on a held-out test window, and optionally
extrapolate beyond all observed data.

Set `N_FUTURE = 0` to skip the extrapolation.

In [ ]:
# ── PARAMETERS ─────────────────────────────────────────────────────
CYL         = "Middle"
DIR         = "extend"
TRAIN_START = "2026-03-05 16:00"
TRAIN_END   = "2026-03-05 18:00"
TEST_END    = "2026-03-05 18:30"
KERNEL      = "rbf"   # "rbf" | "matern52" | "matern32" | "periodic+rbf"
N_RESTARTS  = 3

# Future extrapolation beyond TEST_END
N_FUTURE        = 20   # steps ahead  (0 = skip)
FUTURE_STEP_MIN = 10    # step resolution in minutes
# ───────────────────────────────────────────────────────────────────

r = run_gp(
    cylinder=CYL, direction=DIR,
    train_start=TRAIN_START, train_end=TRAIN_END, test_end=TEST_END,
    kernel=KERNEL, n_restarts=N_RESTARTS, verbose=True,
)

show_results(r)

# ── Optional future extrapolation ───────────────────────────────────
if N_FUTURE > 0:
    t0       = r["t0"]
    t_test   = r["t_test"].ravel()
    model    = r["model"]

    step_sec = FUTURE_STEP_MIN * 60
    t_future = (
        t_test.max() + np.arange(1, N_FUTURE + 1) * step_sec
    ).reshape(-1, 1)
    mu_fut, var_fut = model.predict(t_future)
    mu_fut  = mu_fut.ravel()
    std_fut = np.sqrt(np.clip(var_fut.ravel(), 0, None))
    dt_future = _to_dt(t0, t_future.ravel())

    # Show last 15% of training as context
    t_train  = r["t_train"].ravel()
    y_train  = r["y_train"]
    lookback = max(1, int(len(t_train) * 0.15))
    dt_ctx   = _to_dt(t0, t_train[-lookback:])

    # GP mean over test window for visual continuity
    t_grid = np.linspace(t_test.min(), t_test.max(), 200).reshape(-1, 1)
    mu_grid, _ = model.predict(t_grid)
    dt_grid = _to_dt(t0, t_grid.ravel())

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.scatter(dt_ctx, y_train[-lookback:],
               s=2, alpha=0.25, color="#378ADD", label="Train (last 15%)")
    ax.scatter(_to_dt(t0, r["t_test"].ravel()), r["y_test"],
               s=5, alpha=0.4, color="#1D9E75", label="Test events")
    ax.plot(dt_grid, mu_grid.ravel(), color="#378ADD", lw=1.1, alpha=0.45)
    ax.plot(dt_future, mu_fut, "--", color="#D85A30", lw=2,
            label=f"Forecast (+{N_FUTURE} × {FUTURE_STEP_MIN}min)")
    ax.fill_between(
        dt_future,
        mu_fut - 1.96 * std_fut,
        mu_fut + 1.96 * std_fut,
        color="#D85A30", alpha=0.15, label="95% CI",
    )
    ax.axvline(dt_future[0], color="#888", lw=0.8, linestyle=":")
    ax.set_title(f"Future forecast — {CYL} / {DIR}  [{KERNEL}]", fontsize=12)
    ax.set_ylabel("duration (ms)")
    ax.legend(fontsize=8)
    _fmt_xaxis(ax)
    plt.tight_layout()
    plt.show()

    print(f"\nFirst {min(5, N_FUTURE)} steps:")
    for dt, mu, sd in zip(dt_future[:5], mu_fut[:5], std_fut[:5]):
        print(f"  {dt}  →  {mu:.1f}ms  "
              f"(95% CI: {mu - 1.96*sd:.1f} – {mu + 1.96*sd:.1f})")

## 5b — GP Anomaly Detection

A stroke is flagged as anomalous if its duration falls outside the GP's predictive
interval. Because the CI width adapts to local data density, sparse periods
automatically get wider bands — fewer false positives than a fixed threshold.

In [ ]:
# ── PARAMETERS ─────────────────────────────────────────────────────
CYL         = "Lifting"
DIR         = "extend"
TRAIN_START = "2026-03-05 16:00"
TRAIN_END   = "2026-03-05 18:00"
TEST_END    = "2026-03-05 18:30"
KERNEL      = "matern52"
CI_LEVEL    = 1.96   # 1.96 = 95%  |  2.58 = 99%
# ───────────────────────────────────────────────────────────────────

r = run_gp(
    cylinder=CYL, direction=DIR,
    train_start=TRAIN_START, train_end=TRAIN_END, test_end=TEST_END,
    kernel=KERNEL, verbose=True,
)

t0        = r["t0"]
mean_pred = r["forecast_mean"]
std_pred  = np.sqrt(np.clip(r["forecast_var"], 0, None))
y_test    = r["y_test"]
dt_test   = _to_dt(t0, r["t_test"].ravel())

lo = mean_pred - CI_LEVEL * std_pred
hi = mean_pred + CI_LEVEL * std_pred
anomaly_mask = (y_test < lo) | (y_test > hi)
n_anom = anomaly_mask.sum()

# ── Plot ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

ax.scatter(
    [d for d, a in zip(dt_test, anomaly_mask) if not a],
    y_test[~anomaly_mask],
    s=5, alpha=0.35, color="#378ADD", label="Normal", zorder=2,
)
ax.scatter(
    [d for d, a in zip(dt_test, anomaly_mask) if a],
    y_test[anomaly_mask],
    s=35, alpha=0.9, color="#D85A30", marker="x",
    linewidths=1.2, label=f"Anomaly ({n_anom})", zorder=4,
)
ax.plot(dt_test, mean_pred, color="#555", lw=1.2, label="GP mean", zorder=3)
ax.fill_between(dt_test, lo, hi,
                color="#378ADD", alpha=0.12, label=f"{CI_LEVEL*100/1.96:.0f}% CI")

ax.set_title(
    f"Anomaly detection — {CYL} / {DIR}  "
    f"({n_anom} anomalies = {n_anom / len(y_test) * 100:.1f}% of test strokes)",
    fontsize=12,
)
ax.set_ylabel("duration (ms)")
ax.legend(fontsize=8)
_fmt_xaxis(ax)
plt.tight_layout()
plt.show()

# ── Report ──────────────────────────────────────────────────────────
print(f"\nAnomaly report — {CYL} / {DIR}")
print(f"  Test strokes : {len(y_test):,}")
print(f"  Anomalies    : {n_anom}  ({n_anom / len(y_test) * 100:.1f}%)")
print(f"  CI level     : ±{CI_LEVEL}σ")

if n_anom > 0:
    print(f"\n  Anomalous strokes:")
    for idx in np.where(anomaly_mask)[0]:
        tag = "SLOW" if y_test[idx] > hi[idx] else "FAST"
        print(f"    {dt_test[idx]}  "
              f"actual = {y_test[idx]:.0f}ms  "
              f"expected = {mean_pred[idx]:.0f}ms  "
              f"CI = [{lo[idx]:.0f}, {hi[idx]:.0f}]  → {tag}")
else:
    print("\n  ✓  No anomalies detected in test window.")